# পাঠ 10 - প্রোডাকশনে AI এজেন্ট

এই পাঠে আপনি Microsoft Agent Framework (Python) ব্যবহার করে AI এজেন্টদের জন্য **প্রোডাকশন প্যাটার্ন** শিখবেন। আমরা আলোচনা করব:

- **Observability** — এজেন্ট ইন্টারঅ্যাকশনে টাইমিং এবং লগিং যোগ করা
- **Evaluation** — একটি মূল্যায়নকারী এজেন্ট ব্যবহার করে প্রতিক্রিয়ার গুণমান স্কোর করা
- **Cost management** — টোকেন অপ্টিমাইজেশন এবং মডেল নির্বাচন সংক্রান্ত কৌশলসমূহ

সিনারিওটি একটি **ভ্রমণ এজেন্ট** যা ব্যবহারকারীদের ভ্রমণের পরিকল্পনা করতে সাহায্য করে, এবং এর উপর পর্যবেক্ষণ ও মূল্যায়ন স্তর প্রয়োগ করা আছে।


## সেটআপ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## উৎপাদন বিবেচ্য বিষয়

প্রোটোটাইপ থেকে উৎপাদনে AI এজেন্টগুলো স্থানান্তর করতে তিনটি মূল স্তম্ভের দিকে খুঁটিনাটি মনোযোগ প্রয়োজন:

1. **পর্যবেক্ষণযোগ্যতা** — আপনাকে দেখতে হবে এজেন্ট কী করছে, এটি কত সময় নেয়, এবং এটি কোন কোন টুল কল করে। ট্রেসিং এবং লগিং না থাকলে, উৎপাদনের সমস্যাগুলি ডিবাগ করা প্রায় অসম্ভব।

2. **মূল্যায়ন** — স্বয়ংক্রিয় গুণগত পরীক্ষা নিশ্চিত করে যে সময়ের সাথে এজেন্টের প্রতিক্রিয়াগুলো সঠিক, সম্পূর্ণ, এবং সহায়ক থাকে। একটি মূল্যায়নকারী এজেন্ট নির্ধারিত মানদণ্ডের বিরুদ্ধে প্রতিক্রিয়াগুলোকে স্কোর করতে পারে।

3. **খরচ ব্যবস্থাপনা** — টোকেন ব্যবহার সরাসরি খরচকে প্রভাবিত করে। প্রম্পট অপ্টিমাইজেশন, মডেল নির্বাচন, এবং ক্যাশিং-এর মতো কৌশলগুলো মান বজায় রেখে ব্যয় নিয়ন্ত্রণে রাখতে সাহায্য করে।


## পর্যবেক্ষণযোগ্য এজেন্ট তৈরি করা

আমরা ভ্রমণ টুলগুলি সংজ্ঞায়িত করি এবং টাইমিং দিয়ে এজেন্ট কলটি আবৃত করি যাতে আমরা লেটেন্সি পর্যবেক্ষণ করতে পারি। প্রোডাকশনে আপনি OpenTelemetry বা অনুরূপ কোনো ট্রেসিং ব্যাকএন্ডের সাথে একীভূত করবেন।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## মূল্যায়ন প্যাটার্ন

একটি সাধারণ প্রোডাকশন প্যাটার্ন হল একটি দ্বিতীয় এজেন্টকে **মূল্যায়ক** হিসেবে ব্যবহার করা। মূল্যায়ক প্রাথমিক এজেন্টের প্রতিক্রিয়াকে পূর্বনির্ধারিত মানদণ্ড যেমন সম্পূর্ণতা, সঠিকতা, এবং সহায়কতার বিরুদ্ধে স্কোর দেয়।

এটি সক্ষম করে:
- প্রতিক্রিয়াগুলি ব্যবহারকারীদের কাছে পৌঁছানোর আগে স্বয়ংক্রিয় গুণগতমান গেট
- প্রম্পট বা মডেল পরিবর্তিত হলে রিগ্রেশন সনাক্তকরণ
- সময়ের সাথে এজেন্টের কর্মক্ষমতার ধারাবাহিক পর্যবেক্ষণ


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## খরচ ব্যবস্থাপনার কৌশলসমূহ

Controlling costs is critical for production AI agents. Here are key strategies:

| কৌশল | বর্ণনা |
|---|---|
| **প্রম্পট অপ্টিমাইজেশন** | সিস্টেম নির্দেশনাগুলো সংক্ষেপে রাখুন। ইনপুট টোকেন কমাতে অপ্রয়োজনীয় প্রসঙ্গ সরিয়ে দিন। |
| **মডেল নির্বাচন** | শ্রেণীবিভাগ বা এক্সট্র্যাকশনের মতো সহজ কাজের জন্য ছোট, সস্তা মডেল (e.g. GPT-4o-mini) ব্যবহার করুন, এবং জটিল যুক্তি নির্ণয়ের জন্য বড় মডেল সংরক্ষণ করুন। |
| **ক্যাশিং** | টুলের ফলাফল এবং ঘন ঘন হওয়া কুয়েরিগুলো ক্যাশ করে রাখুন যাতে পুনরাবৃত্ত API কল এড়ানো যায়। |
| **টোকেন বাজেট** | Set `max_tokens` limits to prevent unexpectedly long responses. |
| **ব্যাচিং** | সম্ভাব্য ক্ষেত্রে একাধিক ব্যবহারকারী অনুরোধকে একটি একক API কল-এ গ্রুপ করুন। |

In practice, a tiered approach works well: route straightforward requests to a fast, inexpensive model and escalate only complex queries to a more capable one.


## সারাংশ

এই পাঠে আপনি কীভাবে নিম্নলিখিত কাজগুলো করতে হয় তা শিখেছেন:

1. **পর্যবেক্ষণযোগ্যতা যোগ করা** এজেন্ট ইন্টারঅ্যাকশনে টাইমিং এবং লগিং-এর মাধ্যমে, ট্রেসিং এবং মনিটরিং-এর জন্য ভিত্তি স্থাপন করা।
2. **এজেন্ট প্রতিক্রিয়াগুলি মূল্যায়ন করা** স্বয়ংক্রিয়ভাবে একটি মূল্যায়নকারী এজেন্ট ব্যবহার করে যা সম্পূর্ণতা, নির্ভুলতা এবং সহায়কতা স্কোর করে।
3. **খরচ পরিচালনা করা** প্রম্পট অপ্টিমাইজেশন, মডেল নির্বাচন, ক্যাশিং, এবং টোকেন বাজেটের মাধ্যমে।

এই প্রোডাকশন প্যাটার্নগুলো নিশ্চিত করতে সাহায্য করে যে আপনার AI এজেন্টগুলো বৃহৎ পরিসরে নির্ভরযোগ্য, পরিমাপযোগ্য, এবং খরচ-কার্যকর।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
দায়-অস্বীকৃতি:
এই দলিলটি এআই অনুবাদ সেবা Co-op Translator (https://github.com/Azure/co-op-translator) ব্যবহার করে অনুবাদ করা হয়েছে। আমরা যথাসাধ্য সঠিকতা নিশ্চিত করতে চেষ্টা করি, তবুও স্বয়ংক্রিয় অনুবাদে ভুল বা অসামঞ্জস্য থাকতে পারে। মূল ভাষায় থাকা দলিলটিকেই কর্তৃত্বমূলক উৎস হিসেবে বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের ক্ষেত্রে পেশাদার মানব অনুবাদ গ্রহণ করা সুপারিশ করা হয়। এই অনুবাদের ব্যবহারের ফলে উদ্ভূত কোনো ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
